In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving free_fire_sentiment.csv to free_fire_sentiment.csv


In [ ]:
import pandas as pd

df = pd.read_csv("free_fire_sentiment.csv")

df.head()

,text,sentiment
0,This game's recent ads before matches are gett...,negative
1,The recent nerf to my favorite character ruine...,negative
2,This game has way too many ads before matches ...,negative
3,The recoil pattern on assault rifles feels ran...,negative
4,I love how quick a full match can be finished.,positive


In [ ]:
df.sample()

,text,sentiment
186,Free Fire's new loading screens look genuinely...,positive


In [ ]:
df.isnull().sum()

,0
text,0
sentiment,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.info()

Dataset shape: (300, 2)
Columns: ['text', 'sentiment']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       300 non-null    object
 1   sentiment  300 non-null    object
dtypes: object(2)
memory usage: 4.8+ KB


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
negative,150
positive,150


In [ ]:
# Display sentiment proportions
df["sentiment"].value_counts(normalize=True) * 100

,proportion
sentiment,
negative,50.0
positive,50.0


### Step 2: Converting text into numbers for the RNN.

We'll do this in four small stages:

Normalize the text.

Tokenize each review into words.

Build a vocabulary that maps words to integer IDs.

Convert each review into a sequence of integers.

In [ ]:
import re #to preprocess
from collections import Counter    #to cumt words for building vocab
from sklearn.model_selection import train_test_split

In [ ]:
#normalise and tokenize

def tokenize(text):
    text = text.lower()

    text = re.sub(r"[^a-z0-9\s]", "", text)
    tokens = text.split()

    return tokens

In [ ]:
#lets test on single review

sample_review = df["text"].iloc[0]

print("Original review:", sample_review)
print("Tokens:", tokenize(sample_review))

#perfect now next we will apply it to Df

Original review: This game's recent ads before matches are getting excessive.
Tokens: ['this', 'games', 'recent', 'ads', 'before', 'matches', 'are', 'getting', 'excessive']


In [ ]:
# check if any
df["sentiment"] = df["sentiment"].str.strip().str.lower()


# Convert sentiment labels into numbers
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})


# Tokenize each review
df["tokens"] = df["text"].astype(str).apply(tokenize)

# Split into training and testing sets
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 240
Testing rows: 60


In [ ]:
df["sentiment"].value_counts()

,count
sentiment,
negative,150
positive,150


In [ ]:
df.columns

Index(['text', 'sentiment', 'label', 'tokens'], dtype='object')

In [ ]:
df.head()

,text,sentiment,label,tokens
0,This game's recent ads before matches are gett...,negative,0,"[this, games, recent, ads, before, matches, ar..."
1,The recent nerf to my favorite character ruine...,negative,0,"[the, recent, nerf, to, my, favorite, characte..."
2,This game has way too many ads before matches ...,negative,0,"[this, game, has, way, too, many, ads, before,..."
3,The recoil pattern on assault rifles feels ran...,negative,0,"[the, recoil, pattern, on, assault, rifles, fe..."
4,I love how quick a full match can be finished.,positive,1,"[i, love, how, quick, a, full, match, can, be,..."


In [ ]:
from collections import Counter

words = ["good", "bad", "good", "excellent", "bad", "good"]

word_counts = Counter(words)

print(word_counts)

Counter({'good': 3, 'bad': 2, 'excellent': 1})


In [ ]:
#now train_test_split done now next step is to build a vocab now

#A vocabulary is a dictionary that assigns an integer ID to each word.

# Count words using only  training reviews
word_counts = Counter(
    word
    for tokens in train_df["tokens"]
    for word in tokens
)

# Create vocabulary
word_to_idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

# Assign an ID to each word
for word in word_counts:
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print("Vocabulary size:", len(word_to_idx))
print("Sample vocabulary:", list(word_to_idx.items())[:20])

Vocabulary size: 688
Sample vocabulary: [('<PAD>', 0), ('<UNK>', 1), ('i', 2), ('love', 3), ('diving', 4), ('into', 5), ('the', 6), ('new', 7), ('maps', 8), ('hidden', 9), ('loot', 10), ('spots', 11), ('this', 12), ('patch', 13), ('made', 14), ('mini', 15), ('map', 16), ('update', 17), ('way', 18), ('too', 19)]


In [ ]:
word_to_idx

word_to_idx["most"]

631

<b>1. What is a sequence?

A sequence is simply an ordered collection of elements.

In [ ]:
#Convert reviews into integer sequences

def encode_text(tokens):
    return [
        word_to_idx.get(word, word_to_idx["<UNK>"])
        for word in tokens
    ]

# Convert reviews into integer sequences
train_sequences = train_df["tokens"].apply(encode_text).tolist()
test_sequences = test_df["tokens"].apply(encode_text).tolist()

print("Original tokens:", train_df["tokens"].iloc[0])
print("Encoded sequence:", train_sequences[0])

Original tokens: ['i', 'love', 'diving', 'into', 'the', 'new', 'maps', 'hidden', 'loot', 'spots']
Encoded sequence: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


Next is :<b> Pad the sequences

RNN batches need sequences of equal length. We'll use a maximum sequence length of 40 tokens for this project.

In [ ]:
MAX_LEN = 40

def pad_sequence(sequence, max_len=MAX_LEN):
    # Truncate sequences that are too long
    sequence = sequence[:max_len]

    # Pad shorter sequences with 0
    sequence = sequence + [0] * (max_len - len(sequence))

    return sequence

# Apply padding
X_train = [pad_sequence(seq) for seq in train_sequences]
X_test = [pad_sequence(seq) for seq in test_sequences]

print("First padded sequence:", X_train[0])
print("Sequence length:", len(X_train[0]))

First padded sequence: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sequence length: 40


In [ ]:
#Extract training labels

y_train = train_df["label"].tolist()
y_test = test_df["label"].tolist()

print("First training label:", y_train[0])
print("First testing label:", y_test[0])

First training label: 1
First testing label: 0


<b>Next: Step 3 —

<mark>Convert these sequences and labels into PyTorch tensors and create a Dataset and DataLoader.

In [ ]:
#converting these data into tensors so rnn can understand them

import torch
from torch.utils.data import Dataset, DataLoader

# Convert input sequences into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

# Convert labels into tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)

print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)

X_train shape: torch.Size([240, 40])
X_test shape: torch.Size([60, 40])
y_train shape: torch.Size([240])
y_test shape: torch.Size([60])


In [ ]:
#Creating a Custom PyTorch Dataset

class SentimentDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = SentimentDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = SentimentDataset(
    X_test_tensor,
    y_test_tensor
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 240
Testing samples: 60


In [ ]:
#now finally inspect how dataset will look

sample_x, sample_y = train_dataset[0]

print("Input sequence:", sample_x)
print("Input shape:", sample_x.shape)
print("Label:", sample_y)

Input sequence: tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0])
Input shape: torch.Size([40])
Label: tensor(1.)


In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


#done

In [ ]:
batch_x, batch_y = next(iter(train_loader))

print("Batch input shape:", batch_x.shape)
print("Batch label shape:", batch_y.shape)

print("First review in batch:", batch_x[0])
print("First label in batch:", batch_y[0])

Batch input shape: torch.Size([16, 40])
Batch label shape: torch.Size([16])
First review in batch: tensor([  6,   7, 323, 191,  25, 324, 325,  12,  17,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0])
First label in batch: tensor(1.)


Next step: <b>Step 4 —

<Mark>Build the RNN sentiment classifier using:
Embedding → RNN → Fully Connected Layer → Sentiment Logit

<b>visualizing model

```text
Input token IDs
      ↓
Embedding Layer
      ↓
RNN Layer
      ↓
Final time-step output
      ↓
Fully Connected Layer
      ↓
One logit (sentiment score)

<b>Define the model : </b>

We'll use:
nn.Embedding to convert token IDs into dense vectors.

nn.RNN to process the sequence of word vectors.

nn.Linear to produce one output score per review.

In [ ]:
import torch
import torch.nn as nn

class SentimentRNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_size=64
    ):
        super().__init__()

        # Convert token IDs into embedding vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        # Final classification layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # x shape: (batch_size, sequence_length)
        embedded = self.embedding(x)

        # embedded shape: (batch_size, sequence_length, embedding_dim)
        output, hidden = self.rnn(embedded)

        # Get the final hidden state
        final_hidden = hidden[-1]

        # Generate one logit per review
        logits = self.fc(final_hidden)

        # Shape: (batch_size, 1)
        return logits

In [ ]:
#handle padding correctly

from torch.nn.utils.rnn import pack_padded_sequence

In [ ]:
def forward(self, x):

    # Count non-padding tokens in each review
    lengths = x.ne(0).sum(dim=1).cpu()

    # Convert token IDs into embeddings
    embedded = self.embedding(x)

    # Ignore padding positions during RNN processing
    packed = pack_padded_sequence(
        embedded,
        lengths,
        batch_first=True,
        enforce_sorted=False
    )

    # Process the packed sequence
    _, hidden = self.rnn(packed)

    # Get final hidden state
    final_hidden = hidden[-1]

    # Produce sentiment logit
    logits = self.fc(final_hidden)

    return logits

In [ ]:
#inistolize model

vocab_size = len(word_to_idx)

model = SentimentRNN(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_size=64
)

print(model)

SentimentRNN(
  (embedding): Embedding(688, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [ ]:
# Get one batch from the training DataLoader
batch_x, batch_y = next(iter(train_loader))

# Forward pass
logits = model(batch_x)

print("Input shape:", batch_x.shape)
print("Logits shape:", logits.shape)
print("Logits:\n", logits)

Input shape: torch.Size([16, 40])
Logits shape: torch.Size([16, 1])
Logits:
 tensor([[-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427],
        [-0.1427]], grad_fn=<AddmmBackward0>)


<b>Next: Step 5

<Mark>— Train the RNN using a loss function, optimizer, and training loop.

<b>Step 5 — Train the RNN


Now we'll train our Free Fire Sentiment Classifier.
During training, the model will:

Take a batch of reviews from train_loader.

Generate a logit for each review.

Compare its predictions with the correct labels using a loss function.

Calculate gradients through backpropagation.

Update its parameters using an optimizer.

In [ ]:
import torch
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function and optimizer are ready!")

Loss function and optimizer are ready!


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:

        # Move batch to the selected device
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(batch_x).squeeze(1)

        # Calculate loss
        loss = criterion(logits, batch_y)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10] Loss: 0.6980
Epoch [2/10] Loss: 0.6940
Epoch [3/10] Loss: 0.6935
Epoch [4/10] Loss: 0.6936
Epoch [5/10] Loss: 0.6934
Epoch [6/10] Loss: 0.6941
Epoch [7/10] Loss: 0.6933
Epoch [8/10] Loss: 0.6934
Epoch [9/10] Loss: 0.6935
Epoch [10/10] Loss: 0.6937


<b>Step 6 — Evaluate the Trained RNN


Now we'll evaluate the model on unseen test data to see how well it classifies Free Fire reviews.

We'll calculate:

Test loss
Accuracy
Precision, recall, and F1-score
Confusion matrix

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

all_predictions = []
all_labels = []
total_test_loss = 0

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        logits = model(batch_x).squeeze(1)

        # Calculate loss
        loss = criterion(logits, batch_y)
        total_test_loss += loss.item()

        # Convert logits to probabilities
        probabilities = torch.sigmoid(logits)

        # Convert probabilities to binary predictions
        predictions = (probabilities >= 0.5).long()

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch_y.cpu().long().tolist())

# Calculate metrics
average_test_loss = total_test_loss / len(test_loader)
accuracy = accuracy_score(all_labels, all_predictions)

print(f"Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Test Loss: 0.6932
Test Accuracy: 0.5000

Classification Report:
              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67        30
    Positive       0.00      0.00      0.00        30

    accuracy                           0.50        60
   macro avg       0.25      0.50      0.33        60
weighted avg       0.25      0.50      0.33        60

Confusion Matrix:
[[30  0]
 [30  0]]


#now let's save all the models and files

In [ ]:
import json
import os
import torch

# Create a folder for our saved model files
save_dir = "/content/free_fire_rnn"
os.makedirs(save_dir, exist_ok=True)

# 1. Save model parameters
torch.save(
    model.state_dict(),
    os.path.join(save_dir, "rnn_model.pth")
)

# 2. Save vocabulary
with open(os.path.join(save_dir, "vocabulary.json"), "w") as f:
    json.dump(word_to_idx, f)

# 3. Save model configuration
config = {
    "vocab_size": len(word_to_idx),
    "embedding_dim": 64,
    "hidden_size": 64,
    "max_len": MAX_LEN,
    "padding_id": 0,
    "unknown_id": 1
}

with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=4)

# 4. Save label mapping
label_mapping = {
    "negative": 0,
    "positive": 1
}

with open(os.path.join(save_dir, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=4)

print("Model and preprocessing files saved!")
print("Files:", os.listdir(save_dir))

Model and preprocessing files saved!
Files: ['vocabulary.json', 'rnn_model.pth', 'label_mapping.json', 'config.json']


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/free_fire_rnn_model",
    "zip",
    save_dir
)

print("ZIP created:", zip_path)

ZIP created: /content/free_fire_rnn_model.zip


In [ ]:
from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<b>Step 8: Inference

We'll load the saved model and vocabulary, enter a completely new Free Fire review, and make the model predict:

<div style="
    margin: 30px 0;
    padding: 35px 20px;
    text-align: center;
    background: linear-gradient(135deg, #0B132B, #1C2541, #3A0CA3, #7209B7, #F72585);
    border: 2px solid #F72585;
    border-radius: 20px;
    box-shadow: 0 0 25px rgba(247, 37, 133, 0.45),
                0 8px 20px rgba(0, 0, 0, 0.35);
">

    <h1 style="
        margin: 0;
        font-size: 42px;
        font-weight: 900;
        letter-spacing: 5px;
        color: #FFFFFF;
        text-shadow: 0 0 12px rgba(255,255,255,0.65),
                     0 0 25px rgba(247,37,133,0.8);
    ">
        COMPLETED
    </h1>

    <div style="
        width: 130px;
        height: 5px;
        margin: 18px auto;
        background: linear-gradient(90deg, #00F5D4, #FEE440, #F72585);
        border-radius: 10px;
        box-shadow: 0 0 12px rgba(0,245,212,0.7);
    "></div>

    <p style="
        margin: 0;
        font-size: 15px;
        font-weight: 600;
        letter-spacing: 2px;
        color: #E0FBFC;
    ">
        Project SUCCESSFULLY COMPLETED
    </p>

</div>